# Explorando o pipeline

Notebook de apoio à PoC. Serve para **olhar os dados em cada camada** sem sair do VS Code.

Ordem de uso:

1. rode `python -m src.pipeline` (cria o Bronze) — depois use as seções 1 a 4;
2. rode `dbt run` dentro de `dbt/` (cria Silver e Gold) — depois use as seções 5 e 6.

> Requer o kernel do ambiente do projeto (`.venv` ou `poc`). No VS Code: canto superior direito → *Select Kernel*.

## 0. Preparação

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# a raiz do projeto (este notebook está em notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Raiz do projeto:", RAIZ)

con = duckdb.connect()          # DuckDB em memória: só para consultar arquivos
con.execute(f"set file_search_path='{RAIZ}'")

# Por padrão o pandas TRUNCA a exibição (mostra "..." no meio).
# Aqui pedimos para mostrar tudo — a PoC tem no máximo 408 linhas.
pd.set_option("display.max_rows", None)      # todas as linhas
pd.set_option("display.max_columns", None)   # todas as colunas
pd.set_option("display.width", None)         # sem quebra de largura

def q(sql):
    """Executa SQL e devolve um DataFrame (renderiza bonito no notebook)."""
    return con.execute(sql).df()

## 1. As fontes: o que a origem nos entregou

Antes de qualquer transformação. Repare que **tudo é texto** aqui — CSV não guarda tipo.

In [ ]:
q("select * from 'data/raw/chamados.csv'")

In [ ]:
# o JSON de interações
q("select * from 'data/raw/interacoes.json' limit 10")

## 2. O Bronze: o que o pipeline capturou

Mesmo conteúdo das fontes, em Parquet, com o metadado de ingestão.

In [ ]:
# q("select * from 'data/bronze/chamados.parquet' limit 10")
q("select * from 'data/gold/dim_tempo.parquet' limit 10")

### 2.1 Por que Parquet? Olhe o arquivo por dentro

O `parquet_metadata` mostra **uma linha por coluna** — é a prova de que o formato é colunar.
Compare os tamanhos: colunas com poucos valores distintos (`situacao`) comprimem muito
mais que colunas com valores únicos (`chamado_id`).

In [ ]:
q("""
    select
        path_in_schema            as coluna,
        compression               as compressao,
        total_uncompressed_size   as bytes_sem_compressao,
        total_compressed_size     as bytes_comprimidos
    from parquet_metadata('data/bronze/chamados.parquet')
""")

In [ ]:
# tamanho em disco: CSV x Parquet
csv = (RAIZ / "data/raw/chamados.csv").stat().st_size
pq  = (RAIZ / "data/bronze/chamados.parquet").stat().st_size
print(f"CSV:     {csv:>7,} bytes")
print(f"Parquet: {pq:>7,} bytes")
print(f"\nObs.: com 122 linhas o Parquet pode até ser MAIOR — ele carrega schema e")
print("estatísticas no cabeçalho. A vantagem aparece na escala (milhões de linhas)")
print("e na leitura seletiva de colunas.")

## 3. Caça aos problemas de qualidade

O Bronze preserva o que chegou — **inclusive os defeitos**. Encontre-os aqui
antes de decidir como tratá-los na Silver.

In [ ]:
# a) datas em formatos diferentes?
q("""
    select data_abertura, count(*) as n
    from 'data/bronze/chamados.parquet'
    where data_abertura like '%/%'
    group by 1
""")

In [ ]:
# b) registros duplicados?
q("""
    select chamado_id, count(*) as vezes
    from 'data/bronze/chamados.parquet'
    group by 1
    having count(*) > 1
""")

In [ ]:
# c) identificadores ausentes?
q("""
    select *
    from 'data/bronze/chamados.parquet'
    where chamado_id is null or chamado_id = ''
""")

In [ ]:
# d) nomes de unidade padronizados?
q("select unidade_id, nome_unidade from 'data/bronze/unidades.parquet' order by unidade_id")

In [ ]:
# e) classes: o mesmo id aparece mais de uma vez?
q("select * from 'data/bronze/categorias.parquet' order by categoria_id")

In [ ]:
# f) integridade: existe chamado apontando para unidade inexistente?
q("""
    select p.chamado_id, p.unidade_id
    from 'data/bronze/chamados.parquet' p
    left join 'data/bronze/unidades.parquet' c
           on p.unidade_id = c.unidade_id
    where c.unidade_id is null
""")

## 4. Um resumo do que você encontrou

Anote aqui (em texto mesmo) os problemas e a decisão tomada para cada um.
Essa lista é o roteiro da Silver — e cada linha dela vira, mais adiante, um teste no `schema.yml`.

| # | Problema | Onde | Decisão |
|---|---|---|---|
| 1 | | | |
| 2 | | | |

## 5. Depois do `dbt run`: Silver e Gold

Execute dentro da pasta `dbt/`:

```bash
dbt run
```

E então compare as camadas.

In [ ]:
# Silver: dado tipado e limpo (repare nos TIPOS das colunas!)
q("select * from 'data/silver/stg_chamados.parquet' limit 10")

In [ ]:
# os tipos mudaram do Bronze para a Silver?
bronze = q("select name, type from parquet_schema('data/bronze/chamados.parquet') where name <> 'schema'")
silver = q("select name, type from parquet_schema('data/silver/stg_chamados.parquet') where name <> 'schema'")
bronze.merge(silver, on="name", how="outer", suffixes=("_bronze", "_silver"))

In [ ]:
# Gold: o modelo dimensional
q("select * from 'data/gold/fato_chamado.parquet' limit 10")

## 6. A pergunta de negócio

> Qual é o tempo médio de atendimento por unidade, categoria e período?

A medida `tempo_atendimento_dias` já existe na `fato_chamado` — quem consulta apenas agrupa.

**Resultado esperado:** média geral de **10,8 dias**, com Aquidauana no topo (19,9) e Coxim na base (4,8).

In [ ]:
q("""
    select
        c.nome_unidade,
        cl.nome_categoria,
        t.ano,
        count(*)                               as chamados,
        round(avg(f.tempo_atendimento_dias), 1) as tempo_medio_dias
    from 'data/gold/fato_chamado.parquet' f
    join 'data/gold/dim_unidade.parquet'  c  on f.unidade_sk = c.unidade_sk
    join 'data/gold/dim_categoria.parquet'   cl on f.categoria_sk  = cl.categoria_sk
    join 'data/gold/dim_tempo.parquet'    t  on f.data_abertura_sk = t.data_sk
    where f.tempo_atendimento_dias is not null
    group by 1, 2, 3
    order by 1, 2, 3
""")

### 6.1 Variações — e uma armadilha

O gestor nunca pede só uma coisa. Abaixo: por unidade e ano, por equipe — e o erro que é fácil cometer.

In [ ]:
# por unidade e ano: está melhorando ou piorando?
q("""
    select
        c.nome_unidade,
        t.ano,
        count(*)                                 as chamados_resolvidos,
        round(avg(f.tempo_atendimento_dias), 1)   as tempo_medio_dias
    from 'data/gold/fato_chamado.parquet' f
    join 'data/gold/dim_unidade.parquet' c on f.unidade_sk = c.unidade_sk
    join 'data/gold/dim_tempo.parquet'   t on f.data_abertura_sk = t.data_sk
    where f.tempo_atendimento_dias is not null
    group by 1, 2
    order by 1, 2
""")

In [ ]:
# ⚠ A ARMADILHA: equipe_id é único no estado?
q("""
    select
        equipe_id,
        count(*)                    as chamados,
        count(distinct unidade_id)  as unidades_diferentes
    from 'data/gold/fato_chamado.parquet'
    where tempo_atendimento_dias is not null
    group by 1 order by 1
""")
# Olhe a última coluna: a mesma "equipe 2" aparece em várias unidades.
# Agrupar só por equipe_id somaria a 2ª Equipe de Campo Grande com a de Dourados.

In [ ]:
# por equipe, do jeito certo: SEMPRE com a unidade + significância mínima
q("""
    select
        c.nome_unidade,
        f.equipe_id                                as equipe,
        count(*)                                 as chamados_resolvidos,
        round(avg(f.tempo_atendimento_dias), 1)   as tempo_medio_dias
    from 'data/gold/fato_chamado.parquet' f
    join 'data/gold/dim_unidade.parquet' c on f.unidade_sk = c.unidade_sk
    where f.tempo_atendimento_dias is not null
    group by 1, 2
    having count(*) >= 3
    order by tempo_medio_dias desc
""")

---

**Fechando o raciocínio:** o mesmo dado apareceu quatro vezes neste notebook —
na fonte, no Bronze, na Silver e na Gold. O que mudou a cada passo?

- fonte → Bronze: o **formato**
- Bronze → Silver: o **conteúdo** (qualidade)
- Silver → Gold: a **organização** (modelo)
- Gold → consulta: o **significado** (informação para decidir)